In [ ]:
import polars as pl

daily = pl.read_parquet("../data/1d/1d.parquet").lazy()

In [ ]:
daily.head().collect(engine="streaming")

In [ ]:
data = daily.drop(["close_time", "ignore"]).with_columns(
    pl.col("close").pct_change().over("symbol").alias("return"),
    pl.col("close").pct_change().shift(-1).over("symbol").alias("forward_return")
).collect_schema()

In [ ]:
daily.drop(["close_time", "ignore"]).with_columns(
    pl.col("close").pct_change().over("symbol").alias("return"),
    pl.col("close").pct_change().shift(-1).over("symbol").alias("forward_return"),
).with_columns(
    pl.col("forward_return").rank(descending=True).over("open_time").alias("fwd_rank")
).filter(open_time=pl.datetime(2024,1,1)).sort("fwd_rank").head().collect(engine="streaming")